[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# The Session


## What you will be able to do

Save objects with a session: add new ones, change and delete loaded ones, and say when the SQL for
each change is sent, at a flush, and when it is kept, at a commit. Follow an object from transient
to pending to persistent to detached, make sessions with `sessionmaker`, and let `begin()` commit or
roll back for you. Say what autoflush sends before a query and how to hold it back, and recognize
the id that is `None`, the error raised by a line that only reads, and the change to a JSON
dictionary that no commit ever saves.


## The idea

### The problem

A new student arrives in January. The registrar's office creates a `Student`, enrolls her in three
Spring 2026 sections, and later that week moves her from one section to another and corrects the
spelling of her name. With Core, every one of those is an `insert`, an `update` or a `delete` written
by hand, with the right columns and the right `where`, and the program has to remember what it has
changed and what it has not.

The ORM's session remembers for you. It knows every object it loaded and every object it was given,
notices every attribute that is assigned a new value, and turns all of it into SQL when it is time.
That is also where the surprises are. A new student's id is `None` until the session sends her row,
so the enrollments that need that id cannot be made yet. A query can send an `INSERT` of its own
before its `SELECT`, and fail with an error that belongs to a line written much earlier. And a
change the session never noticed, such as a key set inside a dictionary, is not saved by any number
of commits.

### What a session is

> A **`Session`** is the ORM's workspace. It holds the objects it has loaded or been given, tracks
> every change made to them, and runs one transaction at a time on a connection it borrows.
> **`session.add(obj)`** puts a new object in the session, which makes it **pending**.
> **`session.flush()`** sends every pending change as SQL, `INSERT`, `UPDATE` and `DELETE` in an
> order the foreign keys allow, inside the transaction, and gives new objects their ids, which makes
> them **persistent**. **`session.commit()`** flushes and commits, and **`session.rollback()`**
> undoes everything since the transaction began. **Autoflush** flushes before a query runs, so the
> query sees what the program has already added. A **`sessionmaker`** makes sessions that are
> configured once, for the whole program.

### Why it works that way

- **Changes wait until a flush.** `add()` and assigning an attribute only record the change, so the
  session can send many changes together, in the order the foreign keys require.
- **An id comes from the database.** A new row's id is chosen when its `INSERT` runs, so a pending
  object's id is `None` until the session flushes it.
- **A query flushes first.** Autoflush keeps every query consistent with the program's own changes,
  at the price that the query's line is where an error in one of those changes surfaces.
- **The session sees assignments, not mutations.** `student.program = "History"` is noticed, because
  the ORM watches attributes. A key set inside a `dict` that an attribute holds is invisible to it,
  since the attribute itself still holds the same dictionary.
- **A session runs transactions the way a connection does.** It begins one with its first statement,
  `commit()` and `rollback()` end it, and `begin()` makes a block that commits or rolls back for
  itself, as `engine.begin()` did in the **Connections and Transactions** notebook.
- **An object outlives its session.** When the session closes, its objects are **detached**: they
  keep the values they had, and the session no longer tracks them. **The Identity Map** notebook
  takes that apart.

### Where this shows up

A web service built with FastAPI or Flask usually gives every request a session of its own, made by
a `sessionmaker`, and Flask-SQLAlchemy's `db.session` is exactly that. The **Testing a Data Layer**
notebook binds a session to a connection whose transaction it rolls back after every test, and
**The Identity Map** notebook covers what a session does with the objects it loads. The
**Connections and Transactions** notebook covered the transactions underneath.

### What this notebook covers

- The life of an object: transient, pending, persistent and detached
- `sessionmaker`, and a session that commits or rolls back for you
- Changing and deleting objects, and what a flush sends
- Ids that arrive at a flush
- Autoflush: the query that writes first, and `no_autoflush`
- When to flush, when to commit, and how to make a session
- Admitting a student, finished
- Four errors, from an id that was still `None` to a change that no commit saves

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    zoe = Student(name="Zoe Nakamura")
    session.add(zoe)
    print("added:  ", zoe.id)
    session.flush()
    print("flushed:", zoe.id)
    zoe.name = "Zoe N. Nakamura"
    session.commit()
    print(session.scalars(select(Student.name)).all())
```

```
added:   None
flushed: 1
['Zoe N. Nakamura']
```

The new student had no id until the flush sent her `INSERT`. Renaming her was an assignment, which
the session noticed and turned into an `UPDATE` at the commit, with no `update()` written anywhere.


## Setup

Eleven imports, the college built from its classes, and the engine helper.

- `sqlalchemy` is the library itself, and the cell prints its version
- `Session` and `sessionmaker`, from `sqlalchemy.orm`, make sessions, and `DeclarativeBase`,
  `Mapped` and `mapped_column` map the classes
- `inspect`, from `sqlalchemy`, reports where an object stands with its session, with `select`,
  `func` and `insert`, `create_engine` and `event`, and what the classes need, `MetaData`, `String`,
  `ForeignKey` and the constraints; `JSON` is a column type that Common errors uses
- `IntegrityError`, from `sqlalchemy.exc`, is the error a refused flush raises
- `flag_modified`, from `sqlalchemy.orm.attributes`, and `MutableDict`, from
  `sqlalchemy.ext.mutable`, make a session notice a change inside a dictionary, in Common errors
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

From this notebook on, Setup builds the college from the classes of the **Declarative Models**
notebook: `Student`, `Course`, `Term`, `Section` and `Enrollment`, with `Base.metadata.create_all`
for the tables and an `insert` for every class to load the lists. `PrintStatements` prints what an
engine logs, and `college_engine` is the engine helper.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (JSON, CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event,
                        func, insert, inspect, select)
from sqlalchemy.exc import IntegrityError
from sqlalchemy.ext.mutable import MutableDict
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker
from sqlalchemy.orm.attributes import flag_modified
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    @property
    def level(self):
        """100 for an introductory course, 200 for the next, read from the number in the code."""
        return int(self.code.split("-")[1]) // 100 * 100

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### The life of an object

`state` reports where an object stands with its session. A new student is made, added, flushed and
committed, with `echo` on, so that every statement shows up where it is sent. `echo` is switched on
before the session starts, because a connection decides whether it logs when it is opened:


In [2]:
def state(obj):
    """Where an object stands with its session: transient, pending, persistent, deleted or detached."""
    info = inspect(obj)
    return next(name for name in ("transient", "pending", "persistent", "deleted", "detached") if getattr(info, name))



engine.echo = True
with Session(engine) as session:
    zoe = Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                  started_on=date(2026, 1, 12))
    print("made:     ", state(zoe), "| id", zoe.id)
    session.add(zoe)
    print("added:    ", state(zoe), "| id", zoe.id, "| session.new holds", list(session.new))
    session.flush()
    print("flushed:  ", state(zoe), "| id", zoe.id)
    session.commit()
    print("committed:", state(zoe))
engine.echo = False
print("closed:   ", state(zoe))


made:      transient | id None
added:     pending | id None | session.new holds [Student('Zoe Nakamura', 'Computer Science')]
    BEGIN (implicit)
    INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)
    values: ('Zoe Nakamura', 'znakamura@college.edu', 'Computer Science', '2026-01-12')
flushed:   persistent | id 26
    COMMIT
committed: persistent
closed:    detached


Made, Zoe Nakamura was **transient**: an ordinary object that no session knew about. `add()` made
her **pending** and sent nothing, and `session.new` is the set of objects waiting to be inserted.
The flush began the transaction and sent her `INSERT`, and the database gave her the id 26, which
made her **persistent**. The commit sent `COMMIT`, and closing the session left her **detached**: an
object with her values, which no session tracks any longer.

### sessionmaker, and a session that commits for you

A program makes its sessions from one `sessionmaker`, configured once, instead of passing the engine
everywhere. `SessionLocal.begin()` opens a session and a transaction together, commits when the block
ends, and rolls back if an exception leaves it, exactly as `engine.begin()` does for a connection:


In [3]:
SessionLocal = sessionmaker(engine)

with SessionLocal.begin() as session:
    session.add(Course(code="ART-100", title="Drawing", department="Art", credits=3))

try:
    with SessionLocal.begin() as session:
        session.add(Course(code="ART-110", title="Painting", department="Art", credits=3))
        session.add(Course(code="BIO-101", title="Biology Again", department="Biology", credits=4))
except IntegrityError as error:
    print("refused:", error.orig)

with SessionLocal() as session:
    print(session.scalars(select(Course.code).where(Course.department == "Art")).all())


refused: UNIQUE constraint failed: courses.code
['ART-100']


The first block committed Drawing. The second added two courses, and the second of them repeats the
code `BIO-101`, so its flush was refused and the block rolled back both, Painting included. A plain
`SessionLocal()` is a session with no transaction yet, for reading or for committing by hand.

### Changing and deleting objects, and what a flush sends

A loaded object is changed by assigning to its attributes, and deleted with `session.delete()`. The
session collects the changes in `session.dirty` and `session.deleted`, and the flush turns them into
SQL:


In [4]:
engine.echo = True
with SessionLocal() as session:
    ana = session.scalars(select(Student).where(Student.email == "areyes@college.edu")).one()
    dropped = session.get(Enrollment, (1, 38))                   # Ana Reyes in World History, Spring 2026
    ana.program = "History"
    session.delete(dropped)
    print("dirty:  ", list(session.dirty), "| modified:", session.is_modified(ana))
    print("deleted:", list(session.deleted))
    session.commit()
engine.echo = False


    BEGIN (implicit)
    SELECT students.id, students.name, students.email, students.program, students.started_on
    FROM students
    WHERE students.email = ?
    values: ('areyes@college.edu',)
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: (1, 38)
dirty:   [Student('Ana Reyes', 'History')] | modified: True
deleted: [Enrollment(student 1, section 38, None)]
    UPDATE students SET program=? WHERE students.id = ?
    values: ('History', 1)
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: (1, 38)
    COMMIT


The two `SELECT`s loaded Ana Reyes and her enrollment, and the changes after them went nowhere until
the commit: `session.dirty` held Ana, whose program was assigned, and `session.deleted` the
enrollment. Then the flush sent an `UPDATE` that sets only `program`, the one attribute that
changed, and a `DELETE` that finds the enrollment by its primary key, both columns of it.
`session.get()` loads an object by its primary key, which **The Identity Map** notebook covers. The
loads come first on purpose: a query run after a change would have sent the `UPDATE` early, for a
reason the section on autoflush explains.

### Ids arrive at a flush

A section needs the id of its course, and a new course has none until its `INSERT` runs. `flush()`
sends the course early, inside the transaction, so its id can be used before anything is committed:


In [5]:
with SessionLocal.begin() as session:
    painting = Course(code="ART-110", title="Painting", department="Art", credits=3)
    session.add(painting)
    print("before the flush:", painting.id)
    session.flush()
    print("after the flush: ", painting.id)
    session.add(Section(course_id=painting.id, term_id=4, capacity=18))

ART_SECTIONS = select(Course.code, Section.id, Section.capacity).join(Section).where(Course.department == "Art")
with SessionLocal() as session:
    print(session.execute(ART_SECTIONS).all())


before the flush: None
after the flush:  12
[('ART-110', 41, 18)]


The flush sent the course's `INSERT`, and the database gave it the id 12, the next after Drawing's
11. Nothing was committed until the block ended, so a failure after the flush would still have
undone the course. The **Relationships** notebook removes the need for most of these flushes: a
section that holds its course as an object gets the course's id at the flush that inserts them both.

### Autoflush: the query that writes first

Before a query runs, the session flushes whatever is pending, so the query can see it. Zoe Nakamura
is enrolled in Programming I, section 35, and her enrollments are counted twice, once with autoflush
held back by `no_autoflush`, and once without:


In [6]:
ZOE_ENROLLMENTS = select(func.count()).select_from(Enrollment).where(Enrollment.student_id == 26)

engine.echo = True
with SessionLocal() as session:
    session.add(Enrollment(student_id=26, section_id=35))
    with session.no_autoflush:
        print("counted with autoflush held back:", session.scalar(ZOE_ENROLLMENTS))
    print("counted after autoflush:          ", session.scalar(ZOE_ENROLLMENTS))
    session.rollback()
engine.echo = False


    BEGIN (implicit)
    SELECT count(*) AS count_1
    FROM enrollments
    WHERE enrollments.student_id = ?
    values: (26,)
counted with autoflush held back: 0
    INSERT INTO enrollments (student_id, section_id, grade) VALUES (?, ?, ?) RETURNING status
    values: (26, 35, None)
    SELECT count(*) AS count_1
    FROM enrollments
    WHERE enrollments.student_id = ?
    values: (26,)
counted after autoflush:           1
    ROLLBACK


Inside `no_autoflush`, the `SELECT` went alone, and the database had no enrollment to count. Outside
it, the session sent the pending `INSERT` first, so the same query counted one. The `INSERT` ends
with `RETURNING status`: the table's default supplied the status, and the session asked for it back
to keep the object up to date. The rollback undid the enrollment, and nothing was saved. Autoflush
is on by default, and it is almost always what a program wants; Common errors shows the one way it
surprises.

### When to flush, when to commit, and how to make a session

| Use | When | Why |
|---|---|---|
| `sessionmaker(engine)`, made once | any program with more than one session | the engine and the options are set in one place |
| `with SessionLocal.begin() as session:` | a unit of work that must succeed or fail as one | commits at the end and rolls back on any exception |
| `with SessionLocal() as session:` and `commit()` | reading, or work that commits in stages | the program decides when to commit |
| `session.flush()` | an id, or a database check, needed before the commit | the SQL is sent now, and still inside the transaction |
| `session.no_autoflush` | a query that must not send what is pending, such as a check before an add | the query sees only what the database already has |
| `flag_modified` or `MutableDict` | a change inside a `JSON` dictionary or list | the session sees only assignments |

The default is a `sessionmaker` made once, and `SessionLocal.begin()` around every unit of work, with
a `flush()` wherever an id is needed early.

### Admitting a student, finished

The pieces of this notebook in one function. `admit` makes a student and her enrollments in one
`SessionLocal.begin()` block: the query that finds the Spring 2026 sections sends the student's
`INSERT` first, by autoflush, which gives her an id for the enrollments, and a course with no section
raises, which rolls everything back. The first admission runs with `echo` on:


In [7]:
SPRING_SECTIONS = (
    select(Course.code, Section.id)
    .select_from(Section)
    .join(Course)
    .join(Term)
    .where(Term.name == "Spring 2026")
)


def admit(SessionLocal, name, email, program, codes):
    """Admit a student for Spring 2026 and enroll them in courses by code: all of it, or none of it."""
    with SessionLocal.begin() as session:
        student = Student(name=name, email=email, program=program, started_on=date(2026, 1, 12))
        session.add(student)
        section_of = dict(session.execute(SPRING_SECTIONS).all())     # autoflush sends the student first
        for code in codes:
            if code not in section_of:
                raise LookupError(f"there is no Spring 2026 section of {code}")
            session.add(Enrollment(student_id=student.id, section_id=section_of[code]))
        return student.id



engine.echo = True
omar = admit(SessionLocal, "Omar Said", "osaid@college.edu", "Mathematics", ["MAT-120", "STA-200"])
engine.echo = False
print("admitted student", omar)

try:
    admit(SessionLocal, "Priya Shah", "pshah@college.edu", "Biology", ["BIO-101", "ART-900"])
except LookupError as error:
    print("not admitted:", error)

with SessionLocal() as session:
    print("Priya Shah saved:", session.scalars(select(Student).where(Student.email == "pshah@college.edu")).first())
    print("Omar Said's sections:", session.scalars(select(Enrollment.section_id).where(Enrollment.student_id == omar)).all())


    BEGIN (implicit)
    INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)
    values: ('Omar Said', 'osaid@college.edu', 'Mathematics', '2026-01-12')
    SELECT courses.code, sections.id
    FROM sections JOIN courses ON courses.id = sections.course_id JOIN terms ON terms.id = sections.term_id
    WHERE terms.name = ?
    values: ('Spring 2026',)
    INSERT INTO enrollments (student_id, section_id, grade) VALUES (?, ?, ?), (?, ?, ?) RETURNING status, student_id, section_id
    values: (27, 33, None, 27, 40, None)
    COMMIT
admitted student 27
not admitted: there is no Spring 2026 section of ART-900
Priya Shah saved: None
Omar Said's sections: [33, 40]


The log shows the order: the student's `INSERT`, sent by autoflush when the query for the sections
ran, then the query, then one `INSERT` for both enrollments, and one `COMMIT`. Omar Said is student
27, enrolled in sections 33 and 40. Priya Shah's second course has no section, so `admit` raised,
the block rolled back, and her `INSERT`, which autoflush had already sent, was undone with
everything else: there is no Priya Shah to find.

### Where each part came from

| In `admit` | What it relies on | The section that showed it |
|---|---|---|
| `with SessionLocal.begin() as session:` | commit at the end, rollback on any exception | sessionmaker, and a session that commits for you |
| `session.add(student)` | a pending object, sent at the next flush | The life of an object |
| `session.execute(SPRING_SECTIONS)` | autoflush sending the student before the query | Autoflush: the query that writes first |
| `student.id`, after the query | an id given at the flush | Ids arrive at a flush |
| `raise LookupError(...)` | an exception that rolls back what was flushed | sessionmaker, and a session that commits for you |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/09-the-session-solutions.ipynb).

**1.** Add a student of your own, print their state and id before and after `flush()`, and commit.


In [8]:
# your code here


**2.** With `SessionLocal.begin()`, raise the capacity of Spring 2026's Statistics section, id 40, to
35 by assigning to the attribute, and show the new capacity from a new session.


In [9]:
# your code here


**3.** Load Ben Okafor's enrollment in section 32 with `session.get`, delete it, and print
`session.deleted` before the commit.


In [10]:
# your code here


**4.** In one session, add an enrollment for Zoe Nakamura, student 26, in section 36, and count the
enrollments of section 36 inside `no_autoflush` and outside it. Roll back at the end.


In [11]:
# your code here


**5.** Print what a session holds in `session.new`, `session.dirty` and `session.deleted` after you
add a course, change a student's program and delete an enrollment, then roll back.


In [12]:
# your code here


**6.** Write `withdraw(SessionLocal, student_id, section_id)`, which sets an enrollment's status to
`withdrawn` and returns `True`, or returns `False` when there is no such enrollment. Try it on an
enrollment that exists and on one that does not.


In [13]:
# your code here


## Common errors

### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: sections.course_id


In [14]:
with SessionLocal.begin() as session:
    sculpture = Course(code="ART-120", title="Sculpture", department="Art", credits=3)
    session.add(sculpture)
    session.add(Section(course_id=sculpture.id, term_id=4, capacity=12))


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: sections.course_id
[SQL: INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)]
[parameters: (None, 4, 12)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The section was made while the course was still pending, and a pending object's id is `None`, so the
section's `course_id` was `None` too. The flush at the end of the block inserted the course and then
the section, whose `INSERT`, in the traceback, has `None` as its first value, and the column refused
it. Flush before using the id, as the section on ids does:


In [15]:
with SessionLocal.begin() as session:
    sculpture = Course(code="ART-120", title="Sculpture", department="Art", credits=3)
    session.add(sculpture)
    session.flush()
    session.add(Section(course_id=sculpture.id, term_id=4, capacity=12))
    print("Sculpture is course", sculpture.id)


Sculpture is course 13


### sqlalchemy.exc.IntegrityError: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)


In [16]:
with SessionLocal() as session:
    session.add(Student(name="Ana Reyes", email="areyes@college.edu", program="History", started_on=date(2026, 1, 12)))
    history = session.scalar(select(func.count()).select_from(Student).where(Student.program == "History"))


IntegrityError: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: students.email
[SQL: INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)]
[parameters: ('Ana Reyes', 'areyes@college.edu', 'History', '2026-01-12')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The error came from the second line, a query, and it is about the first. The email belongs to a
student already, and the pending student's `INSERT` was sent by autoflush when the query ran, so the
query's line is where the unique constraint failed. The first line of the message says so, and the
rest names the real failure. The fix is to find out at the line where the mistake is made: flush
right after `add()`, and handle the refusal there:


In [17]:
with SessionLocal() as session:
    session.add(Student(name="Ana Reyes", email="areyes@college.edu", program="History", started_on=date(2026, 1, 12)))
    try:
        session.flush()
    except IntegrityError as error:
        session.rollback()
        print("not added:", error.orig)
    print("History students:", session.scalar(select(func.count()).select_from(Student).where(Student.program == "History")))


not added: UNIQUE constraint failed: students.email
History students: 6


### No error, and the change never saved: a JSON dictionary changed in place


In [18]:
class AdvisingBase(DeclarativeBase):
    pass


class Checklist(AdvisingBase):
    __tablename__ = "checklists"

    id: Mapped[int] = mapped_column(primary_key=True)
    student_id: Mapped[int]
    items: Mapped[dict] = mapped_column(JSON)


advising = college_engine()
AdvisingBase.metadata.create_all(advising)
with Session(advising) as session:
    session.add(Checklist(student_id=1, items={"transcript_reviewed": False, "plan_signed": False}))
    session.commit()

with Session(advising) as session:
    checklist = session.scalars(select(Checklist)).one()
    checklist.items["plan_signed"] = True                        # a change inside the dictionary
    print("the session noticed:", session.is_modified(checklist))
    session.commit()

with Session(advising) as session:
    print("saved:", session.scalars(select(Checklist.items)).one())


the session noticed: False
saved: {'transcript_reviewed': False, 'plan_signed': False}


The dictionary changed and the database did not. The session watches assignments to `items`, and
`checklist.items["plan_signed"] = True` assigns to a key of the dictionary, not to the attribute,
which still holds the same dictionary it held before. So nothing was modified as far as the session
could tell, the commit had nothing to flush, and nothing raised. Either say so with `flag_modified`,
or give the column a type that watches its own contents, `MutableDict.as_mutable(JSON)`:


In [19]:
with Session(advising) as session:
    checklist = session.scalars(select(Checklist)).one()
    checklist.items["plan_signed"] = True
    flag_modified(checklist, "items")                            # tell the session the value changed
    session.commit()
    print("with flag_modified: ", session.scalars(select(Checklist.items)).one())


class WatchedBase(DeclarativeBase):
    pass


class WatchedChecklist(WatchedBase):
    __tablename__ = "watched_checklists"

    id: Mapped[int] = mapped_column(primary_key=True)
    student_id: Mapped[int]
    items: Mapped[dict] = mapped_column(MutableDict.as_mutable(JSON))


WatchedBase.metadata.create_all(advising)
with Session(advising) as session:
    session.add(WatchedChecklist(student_id=1, items={"transcript_reviewed": False, "plan": {"signed": False}}))
    session.commit()
    watched = session.scalars(select(WatchedChecklist)).one()
    watched.items["transcript_reviewed"] = True
    print("MutableDict noticed a key:          ", session.is_modified(watched))
    session.commit()
    watched.items["plan"]["signed"] = True
    print("MutableDict noticed a nested key:   ", session.is_modified(watched))
advising.dispose()


with flag_modified:  {'transcript_reviewed': False, 'plan_signed': True}
MutableDict noticed a key:           True
MutableDict noticed a nested key:    False


`flag_modified` marks the attribute as changed, and the commit saved it. `MutableDict` makes the
dictionary report its own changes, so setting a key was noticed without help. Only its own keys,
though: the dictionary inside it, `plan`, is an ordinary one, and a change there went unnoticed
again. For values nested more than one level deep, `flag_modified` is the dependable answer.

### sqlalchemy.exc.InvalidRequestError: A transaction is already begun on this Session.


In [20]:
with SessionLocal() as session:
    zoe = session.scalars(select(Student).where(Student.id == 26)).one()        # the first query begins a transaction
    with session.begin():
        zoe.program = "Mathematics"


InvalidRequestError: A transaction is already begun on this Session.

The query began a transaction, as the first statement on a session does, so `session.begin()` found
one already open, exactly as `conn.begin()` did in the **Connections and Transactions** notebook. Use
`SessionLocal.begin()` to open the session and its transaction together, so the read and the change
share one transaction:


In [21]:
with SessionLocal.begin() as session:
    zoe = session.scalars(select(Student).where(Student.id == 26)).one()
    zoe.program = "Mathematics"

with SessionLocal() as session:
    print(session.scalars(select(Student).where(Student.id == 26)).one())


Student('Zoe Nakamura', 'Mathematics')


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [22]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `session.add()` makes an object pending, a flush sends its `INSERT` and gives it an id, and a
  commit keeps it; closing the session leaves it detached.
- Changing an attribute and `session.delete()` are recorded, and the flush turns them into `UPDATE`
  and `DELETE`, sending only the columns that changed.
- `sessionmaker` makes sessions for a whole program, and `SessionLocal.begin()` commits a block or
  rolls it back, like `engine.begin()`.
- Autoflush sends pending changes before a query, so an error from an earlier `add()` can surface on
  a query's line, and `no_autoflush` holds it back.
- A session sees assignments, not changes inside a value: `flag_modified` or `MutableDict` for
  `JSON`.


## What is next

**The Identity Map** notebook looks at the objects a session hands out: `Session.get`, two queries
that return one object, and the attribute that expired the moment you committed.


---

&#8592; **Previous:** [Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/08-column-types.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
